In [1]:
"""
Build Pyrome-specific FlamMap lookups

author: maxwell.cook@colostate.edu
"""

import os, sys

from pathlib import Path
from os.path import join
from fb_tools.weather import (
    load_gridmet_csv, build_flammap_scenario_cache,
    load_flammap_scenario_cache
)

# use the current working directory
projdir = Path.cwd().parents[1] # moves up two, outside code directory
print(f"Project directory set to: {projdir}")

# environ vars
proj_crs = 26913  # NAD83 UTM Zone 13N

Project directory set to: /Users/mcc/Library/CloudStorage/Box-Box/MCC/fire_modeling/FM_PythonWrapper


In [2]:
# --- Load the gridMET climatology by Pyrome
gridmet_fp = Path(join(projdir,"data/tabular/raw/weather/gridmet_clim_CO_pyromes.csv"))
# --- Load and inspect
clim = load_gridmet_csv(gridmet_fp) # parses gridmet columns
print(clim.shape)
print("Pyromes :", sorted(clim["pyrome"].unique()))
print("Years   :", sorted(clim["year"].unique()))
print("Columns :", list(clim.columns))

  [load_gridmet_csv] 30,816 rows, 9 pyromes, years 2010–2025  [legacy flat]
(30816, 18)
Pyromes : [np.int64(42), np.int64(43), np.int64(45), np.int64(46), np.int64(47), np.int64(52), np.int64(53), np.int64(56), np.int64(128)]
Years   : [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Columns : ['system:index', 'pyrome', 'date', 'doy', 'erc', 'fm100', 'fm1000', 'pr', 'rmax', 'rmin', 'vpd', 'year', '.geo', 'tmmx_f', 'tmmn_f', 'vpd_pa', 'ws_mph', 'wd_deg']


In [3]:
# --- Build wind scenario by pyrome
from fb_tools.weather import wind_percentiles_from_cell_cache

# --- Build the wind dataframe from HRRR cache
wind_pcts = wind_percentiles_from_cell_cache(
    cache_dir=join(projdir,'data/weather/pyrome_wind'),
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.97],  # default
)
wind_pcts['42']

  [wind_percentiles_from_cell_cache] 10 pyromes × 5 percentiles


{'ws_mph': {'p25': 7.8, 'p50': 12.0, 'p75': 16.2, 'p90': 19.9, 'p97': 24.6},
 'wd_deg_mean': None,
 'n_obs': 1127}

In [4]:

# → {"42": {"ws_mph": {"p25": 8.4, "p50": 12.1, ...}, "wd_deg_mean": None, "n_obs": 1127}, ...}
#
# # Plug directly into scenario builder — format is identical to build_hrrr_wind_percentiles()
# scenario_cache = build_flammap_scenario_cache(
#     df=gridmet_df,
#     wind_percentiles=wind_pcts,
#     out_dir='data/weather/flammap_scenarios',
# )

In [5]:
# CO pyromes span ~37–41°N; 39.5° used for GSI photoperiod calculation.
# GSI uses tmmn_f (min temp), vpd_pa (from GEE export), and daylength.
# wind_direction=-2 = downhill (worst-case); override with -1 (uphill)
# or an explicit azimuth (0–360) as needed.

CACHE_DIR  = Path(join(projdir,"data/weather/flammap/"))

percentiles = [0.25, 0.75, 0.97]

scenarios = build_flammap_scenario_cache(
    clim,
    pyrome_col="pyrome",
    percentiles=percentiles,
    out_dir=CACHE_DIR,
    lat_deg=39.5, # adjust as-needed
    wind_direction=-2, # FlamMap downhill default
    wind_percentiles=wind_pcts,
)
scenarios

  [gridmet] Wrote pyrome_42_flammap.json
  [gridmet] Wrote pyrome_43_flammap.json
  [gridmet] Wrote pyrome_45_flammap.json
  [gridmet] Wrote pyrome_46_flammap.json
  [gridmet] Wrote pyrome_47_flammap.json
  [gridmet] Wrote pyrome_52_flammap.json
  [gridmet] Wrote pyrome_53_flammap.json
  [gridmet] Wrote pyrome_56_flammap.json
  [gridmet] Wrote pyrome_128_flammap.json
  [build_flammap_scenario_cache] 9 pyromes × 3 scenarios (±0.025 ERC band, wind=external)


{'42': {'pyrome_id': '42',
  'percentiles': [0.25, 0.75, 0.97],
  'erc_band': 0.025,
  'wind_direction': -2,
  'wind_speed_source': 'external',
  'scenarios': {'p25': {'FM_1hr': 4.6,
    'FM_10hr': 5.7,
    'FM_100hr': 8.9,
    'FM_herb': 119.4,
    'FM_woody': 116.9,
    'WIND_DIRECTION': -2,
    'erc_quantile_band': [0.225, 0.275],
    'erc_center': 59.7,
    'scenario_doy': 212,
    'n_sample_days': 172,
    'WIND_SPEED': 7.8},
   'p75': {'FM_1hr': 2.9,
    'FM_10hr': 3.6,
    'FM_100hr': 5.4,
    'FM_herb': 63.7,
    'FM_woody': 81.4,
    'WIND_DIRECTION': -2,
    'erc_quantile_band': [0.725, 0.775],
    'erc_center': 86.0,
    'scenario_doy': 180,
    'n_sample_days': 175,
    'WIND_SPEED': 16.2},
   'p97': {'FM_1hr': 1.6,
    'FM_10hr': 2.0,
    'FM_100hr': 3.6,
    'FM_herb': 30.0,
    'FM_woody': 60.0,
    'WIND_DIRECTION': -2,
    'erc_quantile_band': [0.945, 0.995],
    'erc_center': 100.9,
    'scenario_doy': 179,
    'n_sample_days': 176,
    'WIND_SPEED': 24.6}}},
 '43': {

In [6]:
# --- Look up Pyrome